#**PLUTO TRAINEE SPARK**

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from datetime import datetime, timedelta
from pyspark.sql.window import Window
import random

In [0]:
# 1. Датафрейм с сотрудниками
employees_schema = StructType([
    StructField("employee_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("department", StringType(), True),
    StructField("salary", IntegerType(), True),
    StructField("hire_date", DateType(), True),
    StructField("city", StringType(), True)
])

departments = ["IT", "HR", "Finance", "Marketing", "Sales", "Operations"]
cities = ["Moscow", "Saint Petersburg", "Novosibirsk", "Yekaterinburg", "Kazan"]

employees_data = []
for i in range(1, 501):
    employees_data.append((
        i,
        f"Employee_{i}",
        random.choice(departments),
        random.randint(50000, 150000),
        datetime(2020, 1, 1) + timedelta(days=random.randint(0, 1460)),
        random.choice(cities)
    ))

employees_df = spark.createDataFrame(employees_data, employees_schema)

# 2. Датафрейм с продажами
sales_schema = StructType([
    StructField("sale_id", IntegerType(), True),
    StructField("employee_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("amount", IntegerType(), True),
    StructField("sale_date", DateType(), True),
    StructField("region", StringType(), True)
])

products = ["Laptop", "Phone", "Tablet", "Monitor", "Keyboard"]
regions = ["North", "South", "East", "West", "Central"]

sales_data = []
sale_id = 1
for i in range(600):
    sales_data.append((
        sale_id,
        random.randint(1, 500),
        random.choice(products),
        random.randint(1000, 50000),
        datetime(2023, 1, 1) + timedelta(days=random.randint(0, 365)),
        random.choice(regions)
    ))
    sale_id += 1

sales_df = spark.createDataFrame(sales_data, sales_schema)

# 3. Датафрейм с отделами
departments_schema = StructType([
    StructField("dept_id", IntegerType(), True),
    StructField("dept_name", StringType(), True),
    StructField("manager_id", IntegerType(), True),
    StructField("budget", IntegerType(), True)
])

departments_data = [
    (1, "IT", 15, 1000000),
    (2, "HR", 28, 500000),
    (3, "Finance", 42, 800000),
    (4, "Marketing", 67, 900000),
    (5, "Sales", 89, 1200000),
    (6, "Operations", 104, 700000)
]

departments_df = spark.createDataFrame(departments_data, departments_schema)

# Создаем временные представления для SQL
employees_df.createOrReplaceTempView("employees")
sales_df.createOrReplaceTempView("sales")
departments_df.createOrReplaceTempView("departments")

In [0]:
sql_query_1 = """
SELECT 
    department, 
    COUNT(*) as employee_count,
    AVG(salary) as avg_salary,
    MAX(salary) as max_salary,
    MIN(salary) as min_salary
FROM employees 
GROUP BY department 
ORDER BY avg_salary DESC
"""

spark_query_1 = employees_df.groupBy('department')\
    .agg(
        count('*').alias('employee_count'),
        round(avg('salary'),2).alias('avg_salary'),
        max('salary').alias('max_salary'),
        min('salary').alias('min_salary')
    ).orderBy(col('avg_salary').desc())

spark_query_1.show(1, truncate=False, vertical=True)
spark.sql(sql_query_1).show(1, truncate=False, vertical=True)

In [0]:
sql_query_2 = """
SELECT 
    e.employee_id,
    e.name,
    e.department,
    s.product,
    s.amount,
    s.sale_date
FROM employees e
JOIN sales s ON e.employee_id = s.employee_id
WHERE s.amount > 20000 
    AND s.sale_date >= '2023-06-01'
    AND e.department = 'Sales'
ORDER BY s.amount DESC
"""

spark_query_2 = (employees_df.alias('e')\
    .join(sales_df.alias('s'),col('e.employee_id') == col('s.employee_id'),'inner')\
    .filter((col('s.sale_date')>='2023-06-01') & (col('s.amount')>20000) & (col('e.department') == 'Sales'))\
    .select(
        col('e.employee_id'),
        col('e.name'),
        col('e.department'),
        col('s.product'),
        col('s.amount'),
        col('s.sale_date'))\
    .orderBy(col('s.amount').desc()))

spark_query_2.show(1, truncate=False, vertical=True)
spark.sql(sql_query_2).show(1, truncate=False, vertical=True)

In [0]:
sql_query_3 = """
WITH ranked_employees AS (
    SELECT 
        department,
        name,
        salary,
        AVG(salary) OVER (PARTITION BY department) AS avg_salary_by_dept,
        RANK() OVER (PARTITION BY department ORDER BY salary DESC) AS salary_rank
    FROM employees
)
SELECT 
    department,
    name,
    salary,
    (salary - avg_salary_by_dept) AS diff_from_avg,
    salary_rank
FROM ranked_employees
WHERE salary_rank <= 3
"""
window_dept = Window.partitionBy('department').orderBy(col('salary').desc())

spark_query_3 = (employees_df\
               .withColumn('avg_salary_by_dept',avg('salary').over(Window.partitionBy('department')))\
               .withColumn('salary_rank', rank().over(window_dept))
               .filter(col('salary_rank') <= 3)\
               .select(
                    col('department'),
                    col('name'),
                    col('salary'),
                    round(col('salary') - col('avg_salary_by_dept'),2).alias('diff_from_avg'),
                    col('salary_rank')
               ))
         
spark_query_3.show(1, truncate=False, vertical=True)
spark.sql(sql_query_3).show(1, truncate=False, vertical=True)

In [0]:
sql_query_4 = """
SELECT 
    d.dept_name,
    COUNT(DISTINCT e.employee_id) as total_employees,
    COUNT(s.sale_id) as total_sales,
    SUM(s.amount) as total_revenue,
    AVG(s.amount) as avg_sale_amount
FROM departments d
LEFT JOIN employees e ON d.dept_name = e.department
LEFT JOIN sales s ON e.employee_id = s.employee_id
GROUP BY d.dept_name
HAVING total_sales > 0
ORDER BY total_revenue DESC
"""
spark_query_4 = departments_df.alias('d')\
    .join(employees_df.alias('e'), col('d.dept_name')==col('e.department'),'left')\
    .join(sales_df.alias('s'), col('e.employee_id')==col('s.employee_id'),'left')\
    .groupBy('d.dept_name')\
    .agg(
        countDistinct('e.employee_id').alias('total_employees'),
        count('s.sale_id').alias('total_sales'),
        sum('s.amount').alias('total_revenue'),
        round(avg('s.amount'),2).alias('avg_sale_amount')
        )\
    .filter(col('total_sales')>0)\
    .orderBy(col('total_revenue').desc())

spark_query_4.show(1, truncate=False, vertical=True)
spark.sql(sql_query_4).show(1, truncate=False, vertical=True)

In [0]:
sql_query_5 = """
SELECT 
    region,
    product,
    SUM(amount) as total_sales,
    AVG(amount) as avg_sale_amount,
    COUNT(*) as transaction_count,
    SUM(amount) * 100.0 / SUM(SUM(amount)) OVER (PARTITION BY region) as pct_of_region_total
FROM sales
WHERE sale_date BETWEEN '2023-01-01' AND '2023-12-31'
GROUP BY region, product
HAVING COUNT(*) >= 10
ORDER BY region, total_sales DESC
"""
agg_df = (sales_df\
    .filter(col('sale_date').between('2023-01-01','2023-12-31'))\
    .groupBy('region', 'product')\
    .agg(
        sum('amount').alias('total_sales'),
        round(avg('amount'),2).alias('avg_sale_amount'),
        count('*').alias('transaction_count')
    )
    .filter(col('transaction_count') >= 10))

pct_window = Window.partitionBy('region')

spark_query_5 = (
    agg_df\
    .withColumn(
        'pct_of_region_total', round((col('total_sales')*100)/sum('total_sales').over(pct_window),2))\
    .orderBy('region',col('total_sales').desc())
)

spark_query_5.show(1, truncate=False, vertical=True)
spark.sql(sql_query_5).show(1, truncate=False, vertical=True)

In [0]:
sql_query_6 = """
SELECT 
    department,
    COUNT(*) as total_employees,
    SUM(CASE WHEN salary > 100000 THEN 1 ELSE 0 END) as high_earners,
    AVG(CASE WHEN city = 'Moscow' THEN salary ELSE NULL END) as avg_moscow_salary,
    CASE 
        WHEN AVG(salary) > 90000 THEN 'High'
        WHEN AVG(salary) > 70000 THEN 'Medium'
        ELSE 'Low'
    END as salary_level
FROM employees
GROUP BY department
"""

agg_df = employees_df\
    .groupBy('department')\
    .agg(
        count('*').alias('total_employees'),
        sum(
            when(col('salary') > 100000, 1).otherwise(0))\
                .alias('high_earners'),
        round(avg(
            when(col('city') == 'Moscow', col('salary')).otherwise(None)),2).alias('avg_moscow_salary'),
        avg('salary').alias('avg_salary')
        )

spark_query_6 = agg_df\
    .withColumn('salary_level',
                when(col('avg_salary')>90000, 'High')\
                .when(col('avg_salary')>70000, 'Medium')\
                .otherwise('Low'))\
    .select(
        'department',
        'total_employees',
        'high_earners',
        'avg_moscow_salary',
        'salary_level'
    )

spark_query_6.show(1, truncate=False, vertical=True)
spark.sql(sql_query_6).show(1, truncate=False, vertical=True)

In [0]:
sql_query_7 = """
SELECT 
    employee_id,
    name,
    department,
    salary,
    hire_date,
    LAG(salary, 1) OVER (PARTITION BY department ORDER BY hire_date) as prev_salary,
    LEAD(salary, 1) OVER (PARTITION BY department ORDER BY hire_date) as next_salary,
    SUM(salary) OVER (PARTITION BY department ORDER BY hire_date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as running_total
FROM employees
"""

run_win = Window.partitionBy('department').orderBy('hire_date')

agg_df = employees_df\
    .withColumn('prev_salary',lag('salary',1).over(run_win))\
    .withColumn('next_salary',lead('salary',1).over(run_win))\
    .withColumn('running_total',sum('salary').over(run_win.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
spark_query_7 = agg_df\
    .select(
        'employee_id',
        'name',
        'department',
        'salary',
        'hire_date',
        'prev_salary',
        'next_salary',
        'running_total'
    )

spark_query_7.show(1, truncate=False, vertical=True)
spark.sql(sql_query_7).show(1, truncate=False, vertical=True)

In [0]:
sql_query_8 = """
SELECT 
    department,
    name,
    salary,
    hire_date,
    DATEDIFF(CURRENT_DATE(), hire_date) as days_employed,
    CASE 
        WHEN DATEDIFF(CURRENT_DATE(), hire_date) < 365 THEN 'Junior'
        WHEN DATEDIFF(CURRENT_DATE(), hire_date) < 1095 THEN 'Middle'
        ELSE 'Senior'
    END as experience_level,
    YEAR(hire_date) as hire_year,
    MONTH(hire_date) as hire_month
FROM employees
WHERE DATEDIFF(CURRENT_DATE(), hire_date) > 180
ORDER BY days_employed DESC
"""

spark_query_8 = employees_df\
    .withColumn('days_employed', datediff(current_date(), 'hire_date'))\
    .withColumn('experience_level',
                when(col('days_employed') < 365, 'Junior')\
                .when(col('days_employed') < 1095, 'Middle')\
                .otherwise('Senior'))\
    .withColumn('hire_year', year('hire_date'))\
    .withColumn('hire_month', month('hire_date'))\
    .filter(col('days_employed') > 100)\
    .select(
        'department',
        'name',
        'salary',
        'hire_date',
        'days_employed',
        'experience_level',
        'hire_year',
        'hire_month'
    )\
    .orderBy(col('days_employed').desc())

spark_query_8.show(1, truncate=False, vertical=True)
spark.sql(sql_query_8).show(1, truncate=False, vertical=True)

In [0]:
sql_query_9 = """
SELECT 
    e.department,
    e.city,
    COUNT(DISTINCT e.employee_id) as total_employees,
    COUNT(s.sale_id) as total_sales,
    SUM(s.amount) as total_revenue,
    ROUND(AVG(e.salary), 2) as avg_salary,
    SUM(CASE WHEN s.amount > 25000 THEN 1 ELSE 0 END) as high_value_sales,
    ROUND(SUM(CASE WHEN s.amount > 25000 THEN s.amount ELSE 0 END) * 100.0 / NULLIF(SUM(s.amount), 0), 2) as high_value_percentage
FROM employees e
LEFT JOIN sales s ON e.employee_id = s.employee_id
GROUP BY e.department, e.city
HAVING total_sales > 5 AND total_employees >= 3
ORDER BY e.department, total_revenue DESC
"""
spark_query_9 = employees_df.alias('e').join(sales_df.alias('s'), 'employee_id', 'left')\
    .groupby('e.department','e.city')\
    .agg(
        countDistinct('e.employee_id').alias('total_employees'),
        count('s.sale_id').alias('total_sales'),
        sum('s.amount').alias('total_revenue'),
        round(avg('e.salary'),2).alias('avg_salary'),
        sum(when(col('s.amount') > 25000,1)).alias('high_value_sales'),
        round(sum(when(col('s.amount') > 25000, col('s.amount')).otherwise(0)) * 100.0 / nullif(sum('s.amount'), lit(0)),2).alias('high_value_percentage'))\
    .filter((col('total_sales') > lit(5)) & (col('total_employees') >= lit(3)))\
    .orderBy(col('e.department'), col('total_revenue').desc())

spark_query_9.show(1, truncate=False, vertical=True)
spark.sql(sql_query_9).show(1, truncate=False, vertical=True)

In [0]:
sql_query_10 = """
SELECT 
    employee_id,
    name,
    department,
    city,
    salary,
    ROUND(AVG(salary) OVER (PARTITION BY department), 2) as dept_avg_salary,
    ROUND(AVG(salary) OVER (PARTITION BY city), 2) as city_avg_salary,
    RANK() OVER (PARTITION BY department ORDER BY salary DESC) as dept_salary_rank,
    RANK() OVER (PARTITION BY city ORDER BY salary DESC) as city_salary_rank,
    ROUND((salary - AVG(salary) OVER (PARTITION BY department)) * 100.0 / AVG(salary) OVER (PARTITION BY department), 2) as pct_diff_from_dept_avg
FROM employees
WHERE department IN ('IT', 'Sales', 'Finance')
"""
spark_query = employees_df\
    .filter(col('department').isin(['IT','Sales','Finance']))\
    .withColumn('dept_avg_salary', round(avg('salary').over(Window.partitionBy('department')),2))\
    .withColumn('city_avg_salary', round(avg('salary').over(Window.partitionBy('city')),2))\
    .withColumn('dept_salary_rank', rank().over(Window.partitionBy('department').orderBy(col('salary').desc())))\
    .withColumn('city_salary_rank', rank().over(Window.partitionBy('city').orderBy(col('salary').desc())))\
    .withColumn('pct_diff_from_dept_avg', round(((col('salary') - (avg('salary').over(Window.partitionBy('department')))) * 100.0 / avg('salary').over(Window.partitionBy('department'))),2))

spark_query_10 = spark_query.select(
    'employee_id',
    'name',
    'department',
    'city',
    'salary',
    'dept_avg_salary',
    'city_avg_salary',
    'dept_salary_rank',
    'city_salary_rank',
    'pct_diff_from_dept_avg'
)
spark_query_10.show(1, truncate=False, vertical=True)
spark.sql(sql_query_10).show(1, truncate=False, vertical=True)

In [0]:
sql_query_11 = """
WITH base AS (
  SELECT
    region,
    product,
    COUNT(sale_id) AS sales_count,
    AVG(amount) AS avg_sale_amount
  FROM sales
  WHERE sale_date >= '2023-01-01'
  GROUP BY region, product
),
other_products_avg AS (
  SELECT
    region,
    product,
    AVG(amount) AS avg_other_products_in_region
  FROM sales
  WHERE sale_date >= '2023-01-01'
  GROUP BY region, product
),
unique_sellers AS (
  SELECT
    region,
    product,
    COUNT(DISTINCT employee_id) AS unique_sellers_count
  FROM sales
  WHERE sale_date >= '2023-01-01'
  GROUP BY region, product
)
SELECT
  b.region,
  b.product,
  b.sales_count,
  ROUND(b.avg_sale_amount, 2) AS avg_sale_amount,
  ROUND((
    SELECT AVG(amount)
    FROM sales s2
    WHERE s2.region = b.region AND s2.product != b.product AND s2.sale_date >= '2023-01-01'
  ), 2) AS avg_other_products_in_region,
  us.unique_sellers_count
FROM base b
JOIN unique_sellers us
  ON b.region = us.region AND b.product = us.product
WHERE b.sales_count >= 5
ORDER BY b.region, b.avg_sale_amount DESC
"""

s1_df = sales_df\
    .filter(col('sale_date') >= '2023-01-01')\
    .groupBy('region','product')\
    .agg(
        count('sale_id').alias('sales_count'),
        avg('amount').alias('avg_sale_amount')
    )

s2_df = sales_df.filter(
    col('sale_date') >= '2023-01-01'
).groupBy(
    'region', 'product'
).agg(
    avg('amount').alias('avg_other_products_in_region')
)

s3_df = sales_df.filter(
    col('sale_date') >= '2023-01-01'
).groupBy(
    'region', 'product'
).agg(
    countDistinct('employee_id').alias('unique_sellers_count')
)

spark_query_11 = s1_df.alias('s1')\
    .join(s3_df.alias('s3'), (col('s1.region') == col('s3.region')) & (col('s1.product') == col('s3.product')), 'inner')\
    .join(s2_df.alias('s2'), (col('s1.region') == col('s2.region')) & (col('s2.product') != col('s1.product')),'inner')\
    .filter(col('sales_count') >= 5)\
    .select(
        's1.region',
        's1.product',
        's1.sales_count',
        round('s1.avg_sale_amount',2).alias('avg_sale_amount'),
        round('s2.avg_other_products_in_region',2).alias('avg_other_products_in_region'),
        's3.unique_sellers_count'
        )\
    .orderBy('s1.region', col('s1.avg_sale_amount').desc())

spark_query_11.show(1, truncate=False, vertical=True)
spark.sql(sql_query_11).show(1, truncate=False, vertical=True)

In [0]:
sql_query_12 = """
SELECT 
    region,
    product,
    YEAR(sale_date) as sale_year,
    MONTH(sale_date) as sale_month,
    COUNT(*) as monthly_sales,
    SUM(amount) as monthly_revenue,
    LAG(SUM(amount), 1) OVER (PARTITION BY region, product ORDER BY YEAR(sale_date), MONTH(sale_date)) as prev_month_revenue,
    ROUND((SUM(amount) - LAG(SUM(amount), 1) OVER (PARTITION BY region, product ORDER BY YEAR(sale_date), MONTH(sale_date))) * 100.0 / 
          LAG(SUM(amount), 1) OVER (PARTITION BY region, product ORDER BY YEAR(sale_date), MONTH(sale_date)), 2) as growth_percentage
FROM sales
WHERE sale_date >= '2023-01-01'
GROUP BY region, product, YEAR(sale_date), MONTH(sale_date)
HAVING monthly_sales >= 3
ORDER BY region, product, sale_year, sale_month
"""
sal_dat = sales_df.withColumn('sale_year', year('sale_date'))\
    .withColumn('sale_month', month('sale_date'))

month_agg = sal_dat.filter(col('sale_date') >= '2023-01-01')\
    .groupBy('region' , 'product', 'sale_year', 'sale_month')\
    .agg(
        count('*').alias('monthly_sales'),
        sum('amount').alias('monthly_revenue')
    )
    
window_spec = Window.partitionBy('region', 'product')\
    .orderBy('sale_year', 'sale_month')
    

spark_query_12 = month_agg\
    .withColumn('prev_month_revenue', lag('monthly_revenue', 1).over(window_spec))\
    .withColumn('growth_percentage', round((col('monthly_revenue') - col('prev_month_revenue')) * 100.0 / col('prev_month_revenue'), 2))\
    .filter(col('monthly_sales') >= 3)\
    .orderBy('region' , 'product', 'sale_year', 'sale_month')

spark_query_12.show(1, truncate=False, vertical=True)
spark.sql(sql_query_12).show(1, truncate=False, vertical=True)

In [0]:
sql_query_13 = """
WITH employee_performance AS (
    SELECT 
        e.employee_id,
        e.name,
        e.department,
        e.salary,
        e.hire_date,
        COUNT(s.sale_id) as total_sales,
        SUM(s.amount) as total_revenue,
        AVG(s.amount) as avg_sale_amount,
        MAX(s.amount) as max_sale_amount
    FROM employees e
    LEFT JOIN sales s ON e.employee_id = s.employee_id
    GROUP BY e.employee_id, e.name, e.department, e.salary, e.hire_date
),
department_stats AS (
    SELECT 
        department,
        AVG(total_revenue) as avg_department_revenue,
        AVG(total_sales) as avg_department_sales
    FROM employee_performance
    GROUP BY department
)
SELECT 
    ep.*,
    ds.avg_department_revenue,
    ds.avg_department_sales,
    CASE 
        WHEN ep.total_revenue > ds.avg_department_revenue * 1.5 THEN 'Top Performer'
        WHEN ep.total_revenue > ds.avg_department_revenue THEN 'Above Average'
        WHEN ep.total_revenue > 0 THEN 'Below Average'
        ELSE 'No Sales'
    END as performance_category,
    ROUND(ep.total_revenue * 100.0 / ep.salary, 2) as revenue_to_salary_ratio
FROM employee_performance ep
JOIN department_stats ds ON ep.department = ds.department
WHERE ep.total_sales > 0
ORDER BY ep.department, performance_category DESC, total_revenue DESC
"""

emp_perf = employees_df.alias('e').join(sales_df.alias('s'), col('e.employee_id') == col('s.employee_id'), 'left')\
    .groupBy('e.employee_id', 'e.name', 'e.department', 'e.salary', 'e.hire_date')\
    .agg(
        count('s.sale_id').alias('total_sales'),
        sum('s.amount').alias('total_revenue'),
        avg('s.amount').alias('avg_sale_amount'),
        max('s.amount').alias('max_sale_amount')
    )

dep_stats = emp_perf.groupBy('department')\
    .agg(
        avg('total_revenue').alias('avg_department_revenue'),
        avg('total_sales').alias('avg_department_sales')
    )

spark_query_13 = emp_perf.alias('ep').join(dep_stats.alias('ds'), col('ep.department') == col('ds.department'), 'inner')\
    .withColumn('performance_category', when(col('ep.total_revenue') > col('ds.avg_department_revenue') * 1.5, 'Top Performer')\
        .when(col('ep.total_revenue') > col('ds.avg_department_revenue'), 'Above Average')\
        .when(col('ep.total_revenue') > 0, 'Below Average')\
        .otherwise('No Sales'))\
    .withColumn('revenue_to_salary_ratio', round(col('ep.total_revenue') * 100.0 / col('ep.salary'), 2))\
    .filter(col('ep.total_sales') > 0)\
    .select(
        'ep.*',
        'ds.avg_department_revenue',
        'ds.avg_department_sales',
        'performance_category',
        'revenue_to_salary_ratio'
    ).orderBy('ep.department', col('performance_category').desc(), col('total_revenue').desc())

spark_query_13.show(1, truncate=False, vertical=True)
spark.sql(sql_query_13).show(1, truncate=False, vertical=True)

In [0]:
sql_query_14 = """
SELECT 
    product,
    region,
    COUNT(*) as transaction_count,
    SUM(amount) as total_revenue,
    AVG(amount) as avg_transaction_value,
    MIN(amount) as min_transaction_value,
    MAX(amount) as max_transaction_value,
    COUNT(DISTINCT employee_id) as unique_sellers,
    ROUND(SUM(amount) * 100.0 / SUM(SUM(amount)) OVER (PARTITION BY product), 2) as pct_of_product_total,
    ROUND(SUM(amount) * 100.0 / SUM(SUM(amount)) OVER (PARTITION BY region), 2) as pct_of_region_total,
    RANK() OVER (PARTITION BY region ORDER BY SUM(amount) DESC) as region_rank
FROM sales
WHERE sale_date BETWEEN '2023-01-01' AND '2023-12-31'
GROUP BY product, region
HAVING transaction_count >= 5
ORDER BY product, total_revenue DESC
"""
spark_query_14 = sales_df.filter(col('sale_date').between('2023-01-01', '2023-12-31'))\
    .groupBy('product', 'region')\
    .agg(
        count('*').alias('transaction_count'),
        sum('amount').alias('total_revenue'),
        avg('amount').alias('avg_transaction_value'),
        min('amount').alias('min_transaction_value'),
        max('amount').alias('max_transaction_valie'),
        countDistinct('employee_id').alias('unique_sellers')
    ).filter(col('transaction_count') >= 5)\
    .withColumn('pct_of_product_total', round(col('total_revenue') * 100.0 / sum('total_revenue').over(Window.partitionBy('product')), 2))\
    .withColumn('pct_of_region_total', round(col('total_revenue') * 100.0 / sum('total_revenue').over(Window.partitionBy('region')), 2))\
    .withColumn('region_rank', rank().over(Window.partitionBy('region').orderBy(col('total_revenue').desc())))\
    .orderBy('product', col('total_revenue').desc())

spark_query_14.show(1, truncate=False, vertical=True)
spark.sql(sql_query_14).show(1, truncate=False, vertical=True)

In [0]:
simple_sql_query_1 = """
SELECT 
    department,
    city,
    COUNT(*) as employee_count,
    ROUND(AVG(salary), 2) as avg_salary,
    MAX(salary) as max_salary
FROM employees
WHERE salary > 60000
GROUP BY department, city
HAVING COUNT(*) >= 5
ORDER BY department, avg_salary DESC
"""
spark_simple_query_1 = employees_df\
    .filter(col('salary') > 60000)\
    .groupBy('department', 'city')\
    .agg(
        count('*').alias('employee_count'),
        round(avg('salary'), 2).alias('avg_salary'),
        max('salary').alias('max_salary')
    )\
    .filter(col('employee_count') >= 5)\
    .orderBy('department', col('avg_salary').desc())

spark_simple_query_1.show(1, truncate=False, vertical=True)
spark.sql(simple_sql_query_1).show(1, truncate=False, vertical=True)

In [0]:
simple_sql_query_2 = """
SELECT 
    product,
    COUNT(*) as total_sales,
    SUM(amount) as total_revenue,
    AVG(amount) as avg_sale_amount,
    MIN(amount) as min_sale_amount,
    MAX(amount) as max_sale_amount
FROM sales
WHERE amount BETWEEN 5000 AND 40000
GROUP BY product
ORDER BY total_revenue DESC
"""
spark_simple_query_2 = sales_df\
    .filter(col('amount').between(5000, 40000))\
    .groupBy('product')\
    .agg(
        count('*').alias('total_sales'),
        sum('amount').alias('total_revenue'),
        avg('amount').alias('avg_sale_amount'),
        min('amount').alias('min_sale_amount'),
        max('amount').alias('max_sale_amount')
    )\
    .select(
        'product',
        'total_sales',
        'total_revenue',
        'avg_sale_amount',
        'min_sale_amount',
        'max_sale_amount'
    ).orderBy(col('total_revenue').desc())

spark_simple_query_2.show(1, truncate=False, vertical=True)
spark.sql(simple_sql_query_2).show(1, truncate=False, vertical=True)

In [0]:
simple_sql_query_3 = """
SELECT 
    employee_id,
    name,
    department,
    salary,
    city
FROM employees
WHERE department IN ('IT', 'Sales')
ORDER BY salary DESC
LIMIT 20
"""

spark_simple_query_3 = employees_df\
    .filter(col('department').isin(['IT','Sales']))\
    .select(
        'employee_id',
        'name',
        'department',
        'salary',
        'city'
    )\
    .orderBy(col('salary').desc())

print("=== Простой запрос 3 - результат ===")

spark_simple_query_3.show(1, truncate=False, vertical=True)
spark.sql(simple_sql_query_3).show(1, truncate=False, vertical=True)

In [0]:
complex_sql_16 = """
WITH employee_stats AS (
    SELECT 
        e.employee_id,
        e.name,
        e.department,
        e.salary,
        e.hire_date,
        e.city,
        COUNT(s.sale_id) as total_sales,
        SUM(s.amount) as total_revenue,
        AVG(s.amount) as avg_sale_amount,
        MAX(s.amount) as max_sale_amount,
        -- Сложные оконные функции
        PERCENT_RANK() OVER (PARTITION BY e.department ORDER BY SUM(s.amount)) as dept_revenue_percentile,
        CUME_DIST() OVER (PARTITION BY e.department, e.city ORDER BY SUM(s.amount)) as city_dept_revenue_cume,
        NTILE(4) OVER (PARTITION BY e.department ORDER BY SUM(s.amount)) as revenue_quartile
    FROM employees e
    LEFT JOIN sales s ON e.employee_id = s.employee_id
    GROUP BY e.employee_id, e.name, e.department, e.salary, e.hire_date, e.city
),
department_benchmarks AS (
    SELECT
        department,
        AVG(total_revenue) as avg_department_revenue,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY total_revenue) as median_department_revenue,
        STDDEV(total_revenue) as revenue_stddev
    FROM employee_stats
    WHERE total_sales > 0
    GROUP BY department
)
SELECT 
    es.*,
    db.avg_department_revenue,
    db.median_department_revenue,
    db.revenue_stddev,
    (es.total_revenue - db.avg_department_revenue) / NULLIF(db.revenue_stddev, 0) as revenue_z_score,
    CASE 
        WHEN es.total_revenue > db.avg_department_revenue + db.revenue_stddev THEN 'Top Performer'
        WHEN es.total_revenue > db.avg_department_revenue THEN 'Above Average'
        WHEN es.total_revenue > db.avg_department_revenue - db.revenue_stddev THEN 'Below Average'
        ELSE 'Underperformer'
    END as performance_tier,
    LAG(es.total_revenue, 1) OVER (PARTITION BY es.department ORDER BY es.hire_date) as prev_hire_revenue,
    LEAD(es.total_revenue, 1) OVER (PARTITION BY es.department ORDER BY es.hire_date) as next_hire_revenue
FROM employee_stats es
JOIN department_benchmarks db ON es.department = db.department
WHERE es.total_sales > 0
ORDER BY es.department, revenue_z_score DESC
"""
emp_st = employees_df.alias('e').join(sales_df.alias('s'), col('e.employee_id') == col('s.employee_id'), 'left')\
    .groupBy('e.employee_id', 'e.name', 'e.department', 'e.salary', 'e.hire_date', 'e.city')\
    .agg(
        count('s.sale_id').alias('total_sales'),
        sum('s.amount').alias('total_revenue'),
        avg('s.amount').alias('avg_sale_amount'),
        max('s.amount').alias('max_sale_amount')
    ).withColumn('dept_revenue_percentile', percent_rank().over(Window.partitionBy('e.department').orderBy(col('total_revenue'))))\
    .withColumn('city_dept_revenue_cume', cume_dist().over(Window.partitionBy('e.department', 'e.city').orderBy(col('total_revenue'))))\
    .withColumn('revenue_quartile', ntile(4).over(Window.partitionBy('e.department').orderBy(col('total_revenue'))))

filtered = emp_st.filter(col('total_sales') > 0)
agg_df = filtered.groupBy('department')\
    .agg(
        avg('total_revenue').alias('avg_department_revenue'),
        stddev('total_revenue').alias('revenue_stddev')
    )

medians = filtered.groupBy('department')\
    .agg(
        expr('percentile_approx(total_revenue, 0.5)').alias('median_department_revenue')
    )
dep_ben = agg_df.join(medians, 'department', 'inner')

spark_query_16 = emp_st.alias('es').join(dep_ben.alias('db'), 'department', 'inner')\
    .filter(col('es.total_sales') > 0)\
    .withColumn('revenue_z_score', (col('es.total_revenue') - col('db.avg_department_revenue')) / nullif('db.revenue_stddev', expr('0')))\
    .withColumn('performance_tier', when(col('es.total_revenue') > (col('db.avg_department_revenue') + col('db.revenue_stddev')),'Top Performer')\
    .when(col('es.total_revenue') > col('db.avg_department_revenue'), 'Above Average')\
    .when(col('es.total_revenue') > (col('db.avg_department_revenue') - col('db.revenue_stddev')), 'Below Average')\
    .otherwise('Underperformer'))\
    .withColumn('prev_hire_revenue', lag('es.total_revenue',1).over(Window.partitionBy('es.department').orderBy('es.hire_date')))\
    .withColumn('next_hire_revenue', lead('es.total_revenue',1).over(Window.partitionBy('es.department').orderBy('es.hire_date')))\
    .select(
        'es.*',
        'db.avg_department_revenue',
        'db.median_department_revenue',
        'revenue_z_score',
        'performance_tier',
        'prev_hire_revenue',
        'next_hire_revenue'
    ).orderBy('es.department', col('revenue_z_score').desc())

spark_query_16.show(1, truncate=False, vertical=True)
spark.sql(complex_sql_16).show(1, truncate=False,vertical=True)

In [0]:
complex_sql_17 = """
WITH monthly_sales AS (
    SELECT 
        e.department,
        s.region,
        s.product,
        DATE_TRUNC('month', s.sale_date) as sale_month,
        COUNT(*) as sales_count,
        SUM(s.amount) as monthly_revenue,
        COUNT(DISTINCT s.employee_id) as unique_sellers,
        AVG(s.amount) as avg_sale_amount
    FROM sales s
    JOIN employees e ON s.employee_id = e.employee_id
    WHERE s.sale_date >= '2023-01-01'
    GROUP BY e.department, s.region, s.product, DATE_TRUNC('month', s.sale_date)
),
sales_trends AS (
    SELECT 
        *,
        -- Скользящие средние и кумулятивные суммы
        AVG(monthly_revenue) OVER (
            PARTITION BY department, region, product 
            ORDER BY sale_month 
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) as moving_avg_3_months,
        SUM(monthly_revenue) OVER (
            PARTITION BY department, region, product 
            ORDER BY sale_month 
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) as cumulative_revenue,
        -- Анализ роста
        LAG(monthly_revenue, 1) OVER (
            PARTITION BY department, region, product 
            ORDER BY sale_month
        ) as prev_month_revenue,
        LAG(monthly_revenue, 12) OVER (
            PARTITION BY department, region, product 
            ORDER BY sale_month
        ) as prev_year_month_revenue,
        -- Ранжирование внутри групп
        ROW_NUMBER() OVER (
            PARTITION BY department, region 
            ORDER BY monthly_revenue DESC
        ) as revenue_rank_in_region_dept
    FROM monthly_sales
)
SELECT 
    *,
    ROUND((monthly_revenue - prev_month_revenue) * 100.0 / NULLIF(prev_month_revenue, 0), 2) as mom_growth_pct,
    ROUND((monthly_revenue - prev_year_month_revenue) * 100.0 / NULLIF(prev_year_month_revenue, 0), 2) as yoy_growth_pct,
    CASE 
        WHEN monthly_revenue > moving_avg_3_months * 1.2 THEN 'Significant Growth'
        WHEN monthly_revenue > moving_avg_3_months THEN 'Moderate Growth'
        WHEN monthly_revenue < moving_avg_3_months THEN 'Decline'
        ELSE 'Stable'
    END as growth_category,
    PERCENT_RANK() OVER (
        PARTITION BY sale_month 
        ORDER BY monthly_revenue
    ) as monthly_revenue_percentile
FROM sales_trends
WHERE sales_count >= 3
ORDER BY department, region, product, sale_month
"""
sale_withDate = sales_df.withColumn('sale_month', date_trunc('month', 'sale_date'))

mon_sal = sale_withDate.alias('s').join(employees_df.alias('e'), 'employee_id', 'inner')\
    .filter(col('s.sale_date') >= '2023-01-01')\
    .groupBy('e.department', 's.region', 's.product', 's.sale_month')\
    .agg(
        count('*').alias('sales_count'),
        sum('s.amount').alias('monthly_revenue'),
        countDistinct('s.employee_id').alias('unique_sellers'),
        avg('s.amount').alias('avg_sale_amount')
    )

sal_tr = mon_sal.withColumn('moving_avg_3_months', avg('monthly_revenue').over(Window.partitionBy('department', 'region', 'product').orderBy('sale_month').rowsBetween(-2,0)))\
    .withColumn('cumulative_revenue', sum('monthly_revenue').over(Window.partitionBy('department','region','product').orderBy('sale_month').rowsBetween(Window.unboundedPreceding, Window.currentRow)))\
    .withColumn('prev_month_revenue', lag('monthly_revenue', 1).over(Window.partitionBy('department','region','product').orderBy('sale_month')))\
    .withColumn('prev_year_month_revenue', lag('monthly_revenue',12).over(Window.partitionBy('department','region','product').orderBy('sale_month')))\
    .withColumn('revenue_rank_in_region_dept', row_number().over(Window.partitionBy('department','region').orderBy(col('monthly_revenue').desc())))
spark_query_17 = sal_tr.filter(col('sales_count') >= 3)\
    .withColumn('mom_growth_pct', round((col('monthly_revenue') - col('prev_month_revenue')) * 100.0 / nullif(col('prev_month_revenue'), expr('0')), 2))\
    .withColumn('yoy_growth_pct', round((col('monthly_revenue') - col('prev_year_month_revenue')) * 100.0 / nullif(col('prev_year_month_revenue'), expr('0')), 2))\
    .withColumn('growth_category', 
        when(col('monthly_revenue') > col('moving_avg_3_months') * 1.2, 'Significant Growth').when(col('monthly_revenue') > col('moving_avg_3_months'), 'Moderate Growth').when(col('monthly_revenue') < col('moving_avg_3_months'), 'Decline').otherwise('Stable'))\
    .withColumn('monthly_revenue_percentile', percent_rank().over(Window.partitionBy('sale_month').orderBy('monthly_revenue')))\
    .orderBy('department','region','product','sale_month')

spark_query_17.show(1, truncate=False, vertical=True)
spark.sql(complex_sql_17).show(1, truncate=False,vertical=True)

In [0]:
complex_sql_18 = """
WITH department_employee_stats AS (
    SELECT 
        d.dept_id,
        d.dept_name,
        d.budget,
        COUNT(e.employee_id) as total_employees,
        SUM(e.salary) as total_salary_cost,
        AVG(e.salary) as avg_salary,
        COUNT(CASE WHEN e.salary > 100000 THEN 1 END) as high_earners_count
    FROM departments d
    LEFT JOIN employees e ON d.dept_name = e.department
    GROUP BY d.dept_id, d.dept_name, d.budget
),
department_sales_stats AS (
    SELECT 
        e.department,
        COUNT(s.sale_id) as total_sales,
        SUM(s.amount) as total_revenue,
        AVG(s.amount) as avg_sale_amount,
        COUNT(DISTINCT s.employee_id) as active_sellers,
        COUNT(DISTINCT s.product) as unique_products_sold,
        COUNT(DISTINCT s.region) as regions_covered
    FROM employees e
    JOIN sales s ON e.employee_id = s.employee_id
    GROUP BY e.department
),
department_efficiency AS (
    SELECT 
        des.*,
        COALESCE(dss.total_sales, 0) as total_sales,
        COALESCE(dss.total_revenue, 0) as total_revenue,
        COALESCE(dss.avg_sale_amount, 0) as avg_sale_amount,
        COALESCE(dss.active_sellers, 0) as active_sellers,
        COALESCE(dss.unique_products_sold, 0) as unique_products_sold,
        COALESCE(dss.regions_covered, 0) as regions_covered
    FROM department_employee_stats des
    LEFT JOIN department_sales_stats dss ON des.dept_name = dss.department
),
final_metrics AS (
    SELECT 
        *,
        -- Ключевые метрики эффективности
        total_revenue / NULLIF(total_salary_cost, 0) as revenue_per_salary_ratio,
        total_revenue / NULLIF(total_employees, 0) as revenue_per_employee,
        total_sales / NULLIF(total_employees, 0) as sales_per_employee,
        active_sellers * 100.0 / NULLIF(total_employees, 0) as active_sellers_ratio,
        total_revenue / NULLIF(budget, 0) as budget_utilization_ratio,
        -- Сравнение с медианными значениями по всем отделам
        PERCENT_RANK() OVER (ORDER BY total_revenue) as revenue_percentile,
        PERCENT_RANK() OVER (ORDER BY total_revenue / NULLIF(total_employees, 0)) as efficiency_percentile
    FROM department_efficiency
)

SELECT 
    dept_name,
    total_employees,
    active_sellers,
    ROUND(active_sellers_ratio, 2) as active_sellers_pct,
    total_sales,
    total_revenue,
    total_salary_cost,
    budget,
    ROUND(revenue_per_salary_ratio, 2) as roi_ratio,
    ROUND(revenue_per_employee, 2) as revenue_per_emp,
    ROUND(sales_per_employee, 2) as sales_per_emp,
    ROUND(budget_utilization_ratio, 2) as budget_utilization,
    ROUND(revenue_percentile * 100, 2) as revenue_percentile_pct,
    ROUND(efficiency_percentile * 100, 2) as efficiency_percentile_pct,
    CASE 
        WHEN revenue_percentile >= 0.8 AND efficiency_percentile >= 0.8 THEN 'High Revenue & High Efficiency'
        WHEN revenue_percentile >= 0.8 THEN 'High Revenue & Low Efficiency'
        WHEN efficiency_percentile >= 0.8 THEN 'Low Revenue & High Efficiency'
        ELSE 'Low Revenue & Low Efficiency'
    END as performance_quadrant
FROM final_metrics
ORDER BY revenue_per_salary_ratio DESC
"""

dep_emp_stts = departments_df.alias('d').join(employees_df.alias('e'),col('d.dept_name') == col('e.department'), 'left').groupBy('d.dept_id', 'd.dept_name', 'd.budget')\
    .agg(
        count('e.employee_id').alias('total_employees'),
        sum('e.salary').alias('total_salary_cost'),
        avg('e.salary').alias('avg_salary'),
        count(when(col('e.salary') > 100000, 1)).alias('high_earners_count')
    )

dep_sal_stts = employees_df.alias('e').join(sales_df.alias('s'), 'employee_id', 'inner')\
    .groupBy('e.department')\
    .agg(
        count('s.sale_id').alias('total_sales'),
        sum('s.amount').alias('total_revenue'),
        avg('s.amount').alias('avg_sale_amount'),
        countDistinct('s.employee_id').alias('active_sellers'),
        countDistinct('s.product').alias('unique_products_sold'),
        countDistinct('s.region').alias('regions_covered')
    )

dep_eff = dep_emp_stts.alias('des').join(dep_sal_stts.alias('dss'),col('des.dept_name') == col('dss.department'), 'left').select(
    'des.*',
    coalesce(col('dss.total_sales'), lit(0)).alias('total_sales'),
    coalesce(col('dss.total_revenue'), lit(0)).alias('total_revenue'),
    coalesce(col('dss.avg_sale_amount'), lit(0)).alias('avg_sale_amount'),
    coalesce(col('dss.active_sellers'), lit(0)).alias('active_sellers'),
    coalesce(col('dss.unique_products_sold'), lit(0)).alias('unique_products_sold'),
    coalesce(col('dss.regions_covered'), lit(0)).alias('regions_covered')
)

fin_met = dep_eff.withColumn('revenue_per_salary_ratio', col('total_revenue') / nullif(col('total_salary_cost'), lit(0)))\
    .withColumn('revenue_per_employee', col('total_revenue') / nullif(col('total_employees'), lit(0)))\
    .withColumn('sales_per_employee', col('total_sales') / nullif(col('total_employees'), lit(0)))\
    .withColumn('active_sellers_ratio', col('active_sellers') * 100.0 / nullif(col('total_employees'), lit(0)))\
    .withColumn('bidget_utilization_ratio', col('total_revenue') / nullif(col('budget'), lit(0)))\
    .withColumn('revenue_percentile', percent_rank().over(Window.orderBy('total_revenue')))\
    .withColumn('efficiency_percentile', percent_rank().over(Window.orderBy(col('total_revenue') / nullif(col('total_employees'), lit(0)))))

spark_query_18 = fin_met.withColumn('performance_quadrant', when((col('revenue_percentile') >= 0.8) & (col('efficiency_percentile') >= 0.8), 'High Revenue and High Efficiency').when(col('revenue_percentile') >= 0.8, 'High Revenue and Low Efficiency').when(col('efficiency_percentile') >= 0.8, 'Low Revenue and High Efficiency').otherwise('Low Revenue and Low Efficiency'))\
    .select(
    'dept_name',
    'total_employees',
    'active_sellers',
    round(col('active_sellers_ratio'), 2).alias('active_sellers_pct'),
    'total_sales',
    'total_revenue',
    'total_salary_cost',
    'budget',
    round(col('revenue_per_salary_ratio'),2).alias('roi_ratio'),
    round(col('revenue_per_employee'), 2).alias('revenue_emp_per'),
    round(col('sales_per_employee'), 2).alias('sales_per_emp'),
    round(col('bidget_utilization_ratio'), 2).alias('budget_utilization'),
    round(col('revenue_percentile') * 100, 2).alias('revenue_percentile_pct'),
    round(col('efficiency_percentile') * 100, 2).alias('efficiency_percentile_pct'),
    'performance_quadrant'
   ).orderBy(col('revenue_per_salary_ratio').desc())
    
spark_query_18.show(1, truncate=False, vertical=True)
spark.sql(complex_sql_18).show(1, truncate=False,vertical=True)

In [0]:
complex_sql_19 = """
WITH product_region_analysis AS (
    SELECT 
        s.product,
        s.region,
        e.department,
        DATE_PART('quarter', s.sale_date) as sale_quarter,
        COUNT(*) as transaction_count,
        SUM(s.amount) as quarterly_revenue,
        AVG(s.amount) as avg_transaction_value,
        COUNT(DISTINCT s.employee_id) as unique_sellers,
        COUNT(DISTINCT e.city) as cities_covered
    FROM sales s
    JOIN employees e ON s.employee_id = e.employee_id
    WHERE s.sale_date >= '2023-01-01'
    GROUP BY s.product, s.region, e.department, DATE_PART('quarter', s.sale_date)
),
product_growth_metrics AS (
    SELECT 
        *,
        -- Анализ роста в рамках продукта и региона
        LAG(quarterly_revenue, 1) OVER (
            PARTITION BY product, region, department 
            ORDER BY sale_quarter
        ) as prev_quarter_revenue,
        -- Доля продукта в регионе и отделе
        quarterly_revenue * 100.0 / SUM(quarterly_revenue) OVER (
            PARTITION BY region, department, sale_quarter
        ) as pct_of_region_dept_revenue,
        -- Ранжирование продуктов по разным критериям
        RANK() OVER (
            PARTITION BY region, department, sale_quarter 
            ORDER BY quarterly_revenue DESC
        ) as revenue_rank,
        DENSE_RANK() OVER (
            PARTITION BY region, department, sale_quarter 
            ORDER BY avg_transaction_value DESC
        ) as avg_value_rank,
        -- Анализ концентрации продавцов
        unique_sellers * 100.0 / SUM(unique_sellers) OVER (
            PARTITION BY product, sale_quarter
        ) as seller_concentration_pct
    FROM product_region_analysis
    WHERE transaction_count >= 5
),
final_product_strategy AS (
    SELECT 
        *,
        ROUND((quarterly_revenue - prev_quarter_revenue) * 100.0 / NULLIF(prev_quarter_revenue, 0), 2) as qoq_growth_pct,
        -- Анализ стабильности продукта
        CASE 
            WHEN revenue_rank <= 3 AND avg_value_rank <= 3 THEN 'Star Product'
            WHEN revenue_rank <= 3 THEN 'High Revenue - Low Value'
            WHEN avg_value_rank <= 3 THEN 'Low Revenue - High Value'
            WHEN qoq_growth_pct > 20 THEN 'Emerging Product'
            ELSE 'Standard Product'
        END as product_strategy_category,
        -- Анализ рисков концентрации
        CASE 
            WHEN seller_concentration_pct > 50 THEN 'High Concentration Risk'
            WHEN seller_concentration_pct > 30 THEN 'Medium Concentration Risk'
            ELSE 'Low Concentration Risk'
        END as concentration_risk
    FROM product_growth_metrics
)
SELECT 
    product,
    region,
    department,
    sale_quarter,
    transaction_count,
    ROUND(quarterly_revenue, 2) as quarterly_revenue,
    ROUND(avg_transaction_value, 2) as avg_transaction_value,
    unique_sellers,
    ROUND(pct_of_region_dept_revenue, 2) as market_share_pct,
    revenue_rank,
    avg_value_rank,
    ROUND(qoq_growth_pct, 2) as growth_rate_pct,
    product_strategy_category,
    concentration_risk,
    -- Финальное стратегическое решение
    CASE 
        WHEN product_strategy_category = 'Star Product' AND concentration_risk = 'Low Concentration Risk' THEN 'Invest & Expand'
        WHEN product_strategy_category = 'Star Product' THEN 'Maintain & Diversify Sellers'
        WHEN product_strategy_category = 'Emerging Product' THEN 'Test & Monitor'
        WHEN growth_rate_pct < -10 THEN 'Review & Potentially Phase Out'
        ELSE 'Maintain Current Strategy'
    END as recommended_action
FROM final_product_strategy
ORDER BY sale_quarter, region, department, revenue_rank
"""

prod_reg_an = sales_df.withColumn('sale_quarter', floor(month(col('sale_date')) / 3) + 1).alias('s').join(employees_df.alias('e'), 'employee_id', 'inner')\
    .filter(col('s.sale_date') >= '2023-01-01')\
    .groupBy('s.product', 's.region', 'e.department', 's.sale_quarter')\
    .agg(
        count('*').alias('transaction_count'),
        sum('s.amount').alias('quarterly_revenue'),
        avg('s.amount').alias('avg_transaction_value'),
        countDistinct('s.employee_id').alias('unique_sellers'),
        countDistinct('e.city').alias('cities_covered')
    )

prod_gr_met = prod_reg_an.filter(col('transaction_count') >= 5)\
    .withColumn('prev_quarter_revenue', lag('quarterly_revenue', 1).over(Window.partitionBy('product', 'region', 'department').orderBy('sale_quarter')))\
    .withColumn('pct_of_region_dept_revenue', col('quarterly_revenue') * 100.0 / sum('quarterly_revenue').over(Window.partitionBy('region', 'department', 'sale_quarter')))\
    .withColumn('revenue_rank', rank().over(Window.partitionBy('region', 'department', 'sale_quarter').orderBy(col('quarterly_revenue').desc())))\
    .withColumn('avg_value_rank', dense_rank().over(Window.partitionBy('region', 'department', 'sale_quarter').orderBy(col('avg_transaction_value').desc())))\
    .withColumn('seller_concentration_pct', col('unique_sellers') * 100.0 / sum('unique_sellers').over(Window.partitionBy('product', 'sale_quarter')))

fin_prod_str = prod_gr_met\
    .withColumn('qoq_growth_pct', round((col('quarterly_revenue') - col('prev_quarter_revenue')) * 100.0 / nullif(col('prev_quarter_revenue'), lit(0)), 2))\
    .withColumn('product_strategy_category',
        when((col('revenue_rank') <= 3) & (col('avg_value_rank') <= 3), 'Star Product')\
        .when(col('revenue_rank') <= 3, 'High Revenue - Low Value')\
        .when(col('avg_value_rank') <= 3, 'Low Revenue - High Value')\
        .when(col('qoq_growth_pct') > 20, 'Emerging Product')\
        .otherwise('Standard Product'))\
    .withColumn('concentration_risk',
        when(col('seller_concentration_pct') > 50, 'High Concentration Risk')\
        .when(col('seller_concentration_pct') > 30, 'Medium Concentration Risk')\
        .otherwise('Low Concentration Risk'))\
    .withColumn('growth_rate_pct', round(col('qoq_growth_pct'), 2))\
    .withColumn('recommended_action',
        when((col('product_strategy_category') == 'Star Product') & (col('concentration_risk') == 'Low Concentration Risk'), 'Invest & Expand')\
        .when(col('product_strategy_category') == 'Star Product', 'Maintain & Diversify Sellers')\
        .when(col('product_strategy_category') == 'Emerging Product', 'Test & Monitor')\
        .when(col('growth_rate_pct') < -10, 'Review & Potentially Phase Out')\
        .otherwise('Maintain Current Strategy'))\

spark_query_19 = fin_prod_str.select(
    'product',
    'region',
    'department',
    'sale_quarter',
    'transaction_count',
    round(col('quarterly_revenue'), 2).alias('quarterly_revenue'),
    round(col('avg_transaction_value'), 2).alias('avg_transaction_value'),
    'unique_sellers',
    round(col('pct_of_region_dept_revenue'), 2).alias('market_share_pct'),
    'revenue_rank',
    'avg_value_rank',
    'growth_rate_pct',
    'product_strategy_category',
    'concentration_risk',
    'recommended_action'
).orderBy('sale_quarter', 'region', 'department', 'revenue_rank')

spark_query_19.show(1, truncate=False, vertical=True)
spark.sql(complex_sql_19).show(1, truncate=False, vertical=True)

In [0]:
complex_sql_20 = """
WITH employee_tenure AS (
    SELECT 
        employee_id,
        name,
        department,
        city,
        salary,
        hire_date,
        DATEDIFF(CURRENT_DATE(), hire_date) as days_employed,
        FLOOR(DATEDIFF(CURRENT_DATE(), hire_date) / 365.25) as years_employed
    FROM employees
),
employee_performance AS (
    SELECT 
        e.employee_id,
        COUNT(s.sale_id) as career_sales,
        SUM(s.amount) as career_revenue,
        AVG(s.amount) as career_avg_sale,
        MAX(s.amount) as career_max_sale,
        COUNT(DISTINCT YEAR(s.sale_date)) as active_years,
        -- Продажи за последний год
        SUM(CASE WHEN s.sale_date >= DATE_SUB(CURRENT_DATE(), 365) THEN s.amount ELSE 0 END) as last_year_revenue
    FROM employees e
    LEFT JOIN sales s ON e.employee_id = s.employee_id
    GROUP BY e.employee_id
),
career_progression AS (
    SELECT 
        et.*,
        ep.career_sales,
        ep.career_revenue,
        ep.career_avg_sale,
        ep.career_max_sale,
        ep.active_years,
        ep.last_year_revenue,
        -- Анализ компенсации относительно производительности
        et.salary / NULLIF(ep.career_revenue, 0) as salary_to_revenue_ratio,
        -- Сравнение с коллегами по отделу и опыту
        AVG(et.salary) OVER (
            PARTITION BY et.department, et.years_employed
        ) as avg_salary_peer_group,
        AVG(ep.career_revenue) OVER (
            PARTITION BY et.department, et.years_employed
        ) as avg_revenue_peer_group,
        -- Ранжирование по различным метрикам
        PERCENT_RANK() OVER (
            PARTITION BY et.department 
            ORDER BY et.salary
        ) as salary_percentile_dept,
        PERCENT_RANK() OVER (
            PARTITION BY et.department 
            ORDER BY ep.career_revenue
        ) as revenue_percentile_dept
    FROM employee_tenure et
    JOIN employee_performance ep ON et.employee_id = ep.employee_id
    WHERE ep.career_sales > 0
),
final_analysis AS (
    SELECT 
        *,
        -- Анализ справедливости компенсации
        CASE 
            WHEN salary_percentile_dept > revenue_percentile_dept + 0.2 THEN 'Potentially Overpaid'
            WHEN salary_percentile_dept < revenue_percentile_dept - 0.2 THEN 'Potentially Underpaid'
            ELSE 'Fairly Compensated'
        END as compensation_fairness,
        -- Потенциал роста
        CASE 
            WHEN years_employed < 2 AND revenue_percentile_dept > 0.7 THEN 'High Potential - Early Career'
            WHEN years_employed BETWEEN 2 AND 5 AND revenue_percentile_dept > 0.8 THEN 'High Potential - Mid Career'
            WHEN years_employed > 5 AND revenue_percentile_dept > 0.9 THEN 'Top Performer - Senior'
            ELSE 'Standard Performer'
        END as growth_potential,
        -- Рекомендации по развитию
        last_year_revenue / NULLIF(career_avg_sale, 0) as recent_productivity_ratio
    FROM career_progression
)
SELECT 
    employee_id,
    name,
    department,
    city,
    years_employed,
    salary,
    ROUND(career_revenue, 2) as career_revenue,
    ROUND(salary_to_revenue_ratio, 4) as cost_effectiveness_ratio,
    ROUND(salary_percentile_dept * 100, 2) as salary_percentile,
    ROUND(revenue_percentile_dept * 100, 2) as revenue_percentile,
    compensation_fairness,
    growth_potential,
    CASE 
        WHEN compensation_fairness = 'Potentially Underpaid' AND growth_potential LIKE 'High Potential%' THEN 'Priority for Raise/Promotion'
        WHEN compensation_fairness = 'Potentially Overpaid' AND growth_potential = 'Standard Performer' THEN 'Performance Improvement Plan'
        WHEN growth_potential LIKE 'High Potential%' THEN 'Development Program Candidate'
        WHEN recent_productivity_ratio > 1.5 THEN 'Recent High Performer - Monitor'
        ELSE 'Maintain Current Plan'
    END as hr_recommendation
FROM final_analysis
ORDER BY department, growth_potential, compensation_fairness DESC
"""

spark_query_20.show(1, truncate=False, vertical=True)
spark.sql(complex_sql_20).show(1, vertical=True)